# Name That LLM: 4 visões + embeddings Qwen3-Embedding (mistura escolhida por CV)

Gerado por `scripts/build_kaggle_notebook.py` a partir de [kaggle-name-that-llm](https://github.com/pedrohbenetton/kaggle-name-that-llm). Validação: GroupKFold por prompt (balanced accuracy). As visões de estilo (forma, palavras funcionais) e o agrupamento com teto seguem ideias dos notebooks públicos da competição; a implementação é própria.

In [ ]:
# ---- ntl/data.py ----
"""Rótulos oficiais, caminhos e validação dos dados da competição."""


import os
from pathlib import Path

import pandas as pd

LABELS: tuple[str, ...] = (
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
)

# No Kaggle: /kaggle/input/... Local: pasta baixada dentro do projeto (ignorada pelo git).
# Num notebook não existe __file__, por isso o caminho local é opcional.
_LOCAL = (
    str(Path(globals()["__file__"]).resolve().parents[2] / "name-that-llm" / "name-that-llm")
    if "__file__" in globals()
    else ""
)
_CANDIDATES = (
    os.environ.get("NTL_DATA", ""),
    "/kaggle/input/competitions/name-that-llm",
    "/kaggle/input/name-that-llm",
    _LOCAL,
)


def data_dir() -> Path:
    for c in _CANDIDATES:
        if c and (Path(c) / "train.csv").exists():
            return Path(c)
    raise FileNotFoundError("train.csv não encontrado; defina NTL_DATA com a pasta dos dados")


def validate_train(df: pd.DataFrame) -> None:
    falta = {"id", "prompt_id", "text", "model"} - set(df.columns)
    if falta:
        raise ValueError(f"colunas ausentes: {sorted(falta)}")
    if df["id"].duplicated().any():
        raise ValueError("id repetido")
    if df["text"].isna().any() or (df["text"].astype(str).str.strip() == "").any():
        raise ValueError("texto vazio")
    fora = set(df["model"]) - set(LABELS)
    if fora:
        raise ValueError(f"rótulos fora da lista oficial: {sorted(fora)}")


def load_train(path: Path | None = None) -> pd.DataFrame:
    df = pd.read_csv((path or data_dir()) / "train.csv")
    validate_train(df)
    return df


def load_test(path: Path | None = None) -> pd.DataFrame:
    return pd.read_csv((path or data_dir()) / "test.csv")

In [ ]:
# ---- ntl/models.py ----
"""Modelos. Baseline: TF-IDF de palavras + caracteres e regressão logística.
`four_view_lr` acrescenta duas visões de estilo que ignoram o tema: a "forma" das palavras
(maiúscula/minúscula) e as palavras funcionais. As quatro visões e seus pesos vêm do SVC4 dos
notebooks públicos da competição; a implementação aqui é própria."""


import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline


def tfidf_lr(max_word: int = 25000, max_char: int = 35000, C: float = 2.0) -> Pipeline:
    feats = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=max_word, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_features=max_char,
                                 sublinear_tf=True)),
    ])
    return Pipeline([("feats", feats), ("clf", LogisticRegression(C=C, max_iter=2000))])


FUNCTION_WORDS = frozenset(
    ["a", "an", "the", "and", "or", "but", "if", "while", "although", "because", "as", "so", "than", "that", "which", "who", "whom", "whose", "this", "these", "those", "it", "its", "they", "them", "their", "theirs", "he", "him", "his", "she", "her", "hers", "we", "us", "our", "ours", "you", "your", "yours", "i", "me", "my", "mine", "in", "on", "at", "to", "from", "with", "without", "for", "of", "by", "over", "under", "between", "through", "into", "about", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "can", "could", "will", "would", "shall", "should", "may", "might", "must", "not", "no", "nor", "all", "any", "some", "each", "every", "both", "either", "neither", "other", "another", "such", "only", "also", "just", "very", "too", "more", "most", "less", "least", "here", "there", "when", "where", "why", "how", "what", "whether", "then", "now", "still", "yet", "even", "much", "many", "few", "little"]
)
_WORD = re.compile(r"[^\W\d_]+")
_TOKEN = re.compile(r"\w+|[^\w\s]")


def _shape(w: str) -> str:
    head = "X" if w[0].isupper() else "x"
    if len(w) == 1:
        return head
    return head + ("X" if w[1:].isupper() else "x")


def shape_view(text: str) -> str:
    """Troca cada palavra pelo seu formato (Xx, x, XX); mantém pontuação, números e espaços."""
    return _WORD.sub(lambda m: _shape(m[0]), text)


def function_view(text: str) -> str:
    """Mantém palavras funcionais e pontuação; o resto vira CONTENT."""
    return " ".join(
        t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else "CONTENT"
        for t in _TOKEN.findall(text)
    )


VIEWS = (  # nome, analisador, n-gramas, máx. de features, peso
    ("char", "char", (3, 5), 250000, 1.0),
    ("word", "word", (1, 3), 180000, 0.7),
    ("shape", "char", (2, 6), 60000, 0.5),
    ("function", "word", (1, 3), 50000, 0.5),
)
_PRE = {"shape": shape_view, "function": function_view}


def four_view_features(max_features: int | None = None, min_df: int = 2) -> FeatureUnion:
    parts, weights = [], {}
    for name, analyzer, ngrams, cap, w in VIEWS:
        opts = {"analyzer": analyzer, "ngram_range": ngrams, "lowercase": False,
                "sublinear_tf": True, "min_df": min_df, "max_features": max_features or cap,
                "preprocessor": _PRE.get(name)}
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        parts.append((name, TfidfVectorizer(**opts)))
        weights[name] = w
    return FeatureUnion(parts, transformer_weights=weights)


def four_view_lr(max_features: int | None = None, C: float = 4.0, min_df: int = 2) -> Pipeline:
    return Pipeline([("feats", four_view_features(max_features, min_df)),
                     ("clf", LogisticRegression(C=C, max_iter=3000))])

In [ ]:
# ---- ntl/groups.py ----
"""Infere os grupos de prompt só pelo conteúdo: textos que respondem ao mesmo prompt
compartilham vocabulário de tema. Usa TF-IDF de palavras (sem stopwords) e agrupamento
hierárquico por cosseno."""


from collections.abc import Sequence

import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


def infer_groups(
    texts: Sequence[str], n_groups: int | None = None, per_group: int = 18
) -> np.ndarray:
    if len(texts) < 2:
        return np.zeros(len(texts), dtype=int)
    if n_groups is None:
        n_groups = max(1, round(len(texts) / per_group))
    X = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=1).fit_transform(texts)
    # distância de cosseno pré-computada (n x n): evita densificar a matriz n x vocabulário
    D = np.clip(1.0 - (X @ X.T).toarray(), 0.0, 2.0)
    np.fill_diagonal(D, 0.0)
    n_groups = min(n_groups, len(texts))
    clu = AgglomerativeClustering(n_clusters=n_groups, metric="precomputed", linkage="average")
    return clu.fit_predict(D)


def capped_groups(
    texts: Sequence[str], cap: int = 18, threshold: float = 0.05, max_df: float = 1.0
) -> np.ndarray:
    """Agrupamento hierárquico (ligação média) que nunca forma grupo maior que `cap` e para
    quando a melhor similaridade média cai abaixo de `threshold`. Diferente de `infer_groups`,
    não supõe n/18 grupos: grupos parciais (teste desbalanceado) não são fundidos à força.
    Ideia do teto + limiar vista nos notebooks públicos da competição; implementação própria."""
    n = len(texts)
    if n < 2:
        return np.zeros(n, dtype=int)
    try:
        X = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=max_df).fit_transform(
            texts
        )
    except ValueError:  # vocabulário vazio
        return np.arange(n)
    A = (X @ X.T).toarray().astype(np.float32)
    np.fill_diagonal(A, -np.inf)
    size = np.ones(n, dtype=np.int64)
    alive = np.ones(n, dtype=bool)
    lab = np.arange(n)

    def row_best(k: int) -> tuple[float, int]:
        ok = alive & (size + size[k] <= cap)
        ok[k] = False
        if not ok.any():
            return -np.inf, -1
        r = np.where(ok, A[k], -np.inf)
        j = int(np.argmax(r))
        return float(r[j]), j

    best = np.full(n, -np.inf, dtype=np.float64)
    arg = np.full(n, -1)
    for k in range(n):
        best[k], arg[k] = row_best(k)

    while True:
        i = int(np.argmax(best))
        if not np.isfinite(best[i]) or best[i] < threshold:
            break
        j = int(arg[i])
        si, sj = size[i], size[j]
        new = (si * A[i] + sj * A[j]) / (si + sj)
        A[i], A[:, i] = new, new
        A[j], A[:, j] = -np.inf, -np.inf
        A[i, i] = -np.inf
        size[i] += sj
        alive[j] = False
        best[j], arg[j] = -np.inf, -1
        lab[lab == j] = i
        # linhas cujo melhor parceiro sumiu ou cresceu demais: recalcula
        for k in np.flatnonzero(alive & ((arg == i) | (arg == j))):
            best[k], arg[k] = row_best(int(k))
        best[i], arg[i] = row_best(i)
        # o grupo novo pode virar o melhor parceiro de outras linhas
        cand = alive & (size + size[i] <= cap) & (A[:, i] > best)
        cand[i] = False
        best[cand], arg[cand] = A[cand, i], i
    return np.unique(lab, return_inverse=True)[1]

In [ ]:
# ---- ntl/post.py ----
"""Pós-processamento por grupo de prompt. Cada prompt costuma ter um texto por modelo, então
(1) centralizar os scores dentro do grupo tira o viés de tema e (2) a atribuição (Hungarian)
impõe no máximo um texto por modelo no grupo."""


from collections.abc import Sequence

import numpy as np
from scipy.optimize import linear_sum_assignment


def _groups(groups: Sequence) -> dict:
    idx: dict = {}
    for i, g in enumerate(groups):
        idx.setdefault(g, []).append(i)
    return idx


def center_by_group(scores: np.ndarray, groups: Sequence) -> np.ndarray:
    out = scores.astype(float).copy()
    for rows in _groups(groups).values():
        out[rows] -= out[rows].mean(axis=0)
    return out


def assign_by_group(scores: np.ndarray, groups: Sequence) -> np.ndarray:
    """Devolve índices de classe. Grupos maiores que o nº de classes ficam com o argmax."""
    pred = scores.argmax(axis=1)
    for rows in _groups(groups).values():
        if len(rows) > scores.shape[1]:
            continue
        r, c = linear_sum_assignment(-scores[rows])
        pred[np.asarray(rows)[r]] = c
    return pred

In [ ]:
# ---- ntl/submission.py ----
"""Arquivo de submissão: as mesmas regras que o Kaggle usa para rejeitar."""


from collections.abc import Sequence
from pathlib import Path

import pandas as pd



def make_submission(ids: Sequence, preds: Sequence) -> pd.DataFrame:
    if len(ids) != len(preds):
        raise ValueError("ids e previsões com tamanhos diferentes")
    s = pd.DataFrame({"id": list(ids), "model": list(preds)})
    if s.isna().any().any():
        raise ValueError("valores nulos")
    if s["id"].duplicated().any():
        raise ValueError("id repetido")
    fora = set(s["model"]) - set(LABELS)
    if fora:
        raise ValueError(f"rótulos inválidos: {sorted(fora)}")
    return s


def write_submission(s: pd.DataFrame, out_dir: Path | str = ".") -> Path:
    p = Path(out_dir) / "submission.parquet"
    s.to_parquet(p, index=False)
    return p

In [ ]:
# ---- ntl/cv.py ----
"""Validação cruzada. O teste usa prompts que não estão no treino, então a validação
honesta agrupa por prompt_id (GroupKFold); a aleatória existe só para medir o vazamento."""


from collections.abc import Callable

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import GroupKFold, StratifiedKFold

Folds = list[tuple[np.ndarray, np.ndarray]]


def grouped_folds(df: pd.DataFrame, n_splits: int = 5) -> Folds:
    return list(GroupKFold(n_splits=n_splits).split(df, df["model"], groups=df["prompt_id"]))


def random_folds(df: pd.DataFrame, n_splits: int = 5, seed: int = 0) -> Folds:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(df, df["model"]))


def evaluate(factory: Callable[[], object], df: pd.DataFrame, folds: Folds) -> dict:
    """Treina um modelo novo por fold e mede balanced accuracy (a métrica oficial)."""
    X, y = df["text"].to_numpy(), df["model"].to_numpy()
    oof = np.empty(len(df), dtype=object)
    scores = None
    per_fold = []
    for tr, va in folds:
        m = factory()
        m.fit(X[tr], y[tr])
        oof[va] = m.predict(X[va])
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X[va])
            if scores is None:
                scores = np.zeros((len(df), p.shape[1]))
                classes = list(m.classes_)
            scores[va] = p
        per_fold.append(balanced_accuracy_score(y[va], oof[va]))
    labels = sorted(set(y))
    recalls = recall_score(y, oof, labels=labels, average=None, zero_division=0)
    return {
        "balanced_accuracy": float(balanced_accuracy_score(y, oof)),
        "std": float(np.std(per_fold)),
        "per_fold": [float(x) for x in per_fold],
        "per_class_recall": dict(zip(labels, map(float, recalls), strict=True)),
        "oof": oof,
        "oof_proba": scores,
        "classes": classes if scores is not None else None,
    }


def oof_proba(factory: Callable[[], object], X, y: np.ndarray, folds: Folds):
    """Probabilidades fora do fold para qualquer X indexável (textos ou matriz)."""
    P, classes = None, None
    for tr, va in folds:
        m = factory()
        m.fit(X[tr], y[tr])
        p = m.predict_proba(X[va])
        if P is None:
            P, classes = np.zeros((len(y), p.shape[1])), np.array(m.classes_)
        P[va] = p
    return P, classes

In [ ]:
# ---- ntl/embed.py ----
"""Embeddings de um LLM congelado (mean pooling de algumas camadas), para usar como features.
No Kaggle o modelo vem como Input (sem internet); torch/transformers só são importados ao
extrair, então o resto do pacote roda sem eles."""


from collections.abc import Sequence
from pathlib import Path

import numpy as np


def masked_mean(h, mask):
    """Média dos estados ocultos só nos tokens reais. h: (b, t, d), mask: (b, t).
    Funciona com numpy e com tensores torch."""
    m = mask[..., None] * 1.0
    return (h * m).sum(1) / m.sum(1)


def layer_indices(n_layers: int, fracs: Sequence[float]) -> list[int]:
    return sorted({min(n_layers, max(1, round(f * n_layers))) for f in fracs})


def find_model_dir(root: str | Path, pattern: str) -> Path | None:
    """Primeira pasta sob `root` com config.json cujo caminho contém `pattern`."""
    for cfg in sorted(Path(root).rglob("config.json")):
        if pattern in str(cfg.parent):
            return cfg.parent
    return None


def embed_texts(
    texts: Sequence[str],
    model_dir: str | Path,
    fracs: Sequence[float] = (0.5, 0.75, 1.0),
    max_len: int = 512,
    batch: int = 16,
) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.float16 if dev == "cuda" else torch.float32
    model = AutoModel.from_pretrained(model_dir, torch_dtype=dtype).to(dev).eval()
    idx = layer_indices(model.config.num_hidden_layers, fracs)
    order = np.argsort([len(t) for t in texts])  # lotes de tamanho parecido: menos padding
    out = []
    with torch.no_grad():
        for s in range(0, len(texts), batch):
            chunk = [texts[i] for i in order[s : s + batch]]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len,
                      return_tensors="pt").to(dev)
            hs = model(**enc, output_hidden_states=True).hidden_states
            feats = [masked_mean(hs[i].float(), enc["attention_mask"]) for i in idx]
            out.append(torch.cat(feats, 1).cpu().numpy())
    E = np.empty((len(texts), out[0].shape[1]), dtype=np.float32)
    E[order] = np.concatenate(out)
    del model
    if dev == "cuda":
        torch.cuda.empty_cache()
    return E


def fake_embed(texts: Sequence[str], dim: int = 64) -> np.ndarray:
    """Só para testar o notebook sem GPU: SVD de TF-IDF de caracteres."""
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import TfidfVectorizer

    X = TfidfVectorizer(analyzer="char", ngram_range=(2, 4)).fit_transform(texts)
    return TruncatedSVD(min(dim, X.shape[1] - 1), random_state=0).fit_transform(X)

In [ ]:
# ---- ntl/blend.py ----
"""Classificador sobre embeddings e mistura de log-probabilidades de modelos diferentes."""


from collections.abc import Sequence

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def emb_lr(C: float = 0.05) -> Pipeline:
    return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(C=C, max_iter=3000))])


def blend_logp(Ls: Sequence[np.ndarray], weights: Sequence[float]) -> np.ndarray:
    return sum(w * L for w, L in zip(weights, Ls, strict=True))


def choose_weight(La, Lb, y, classes, grid=None) -> tuple[float, float]:
    """Peso w de La (1-w para Lb) com maior balanced accuracy; empate fica com o maior w."""
    grid = np.round(np.linspace(0, 1, 11), 2) if grid is None else grid
    best = (-1.0, 0.0)
    for w in grid:
        p = np.asarray(classes)[blend_logp([La, Lb], [w, 1 - w]).argmax(1)]
        s = float(balanced_accuracy_score(y, p))
        if s >= best[0]:
            best = (s, float(w))
    return best[1], best[0]


def choose_weights(Ls, y, classes, step: float = 0.1) -> tuple[tuple[float, ...], float]:
    """Pesos (somando 1, passo `step`) da mistura de log-probabilidades com maior balanced
    accuracy. Um modelo que piora a mistura recebe peso 0, então ele não atrapalha."""
    k = round(1 / step)
    classes = np.asarray(classes)
    best: tuple[float, tuple[float, ...]] = (-1.0, ())

    def grid(n, left):
        if n == 1:
            yield (left,)
            return
        for i in range(left + 1):
            for rest in grid(n - 1, left - i):
                yield (i, *rest)

    for combo in grid(len(Ls), k):
        w = tuple(c / k for c in combo)
        s = float(balanced_accuracy_score(y, classes[blend_logp(Ls, w).argmax(1)]))
        if s > best[0]:
            best = (s, w)
    return best[1], best[0]

In [ ]:
# ---- ntl/finetune.py ----
"""Fine-tuning de um LLM de embeddings (Qwen3-Embedding) como classificador de 18 classes:
LoRA nas projeções de atenção e MLP, mean pooling da última camada e uma cabeça linear.
Na GPU (T4) usa fp16 com autocast e GradScaler; sem `peft`, treina só as 2 últimas camadas.
torch/transformers só são importados aqui dentro (o resto do pacote não precisa deles)."""


import math
import time
from collections.abc import Callable, Sequence
from pathlib import Path

import numpy as np


LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def _build(model_dir, n_classes, device, use_lora, lora_r, grad_ckpt):
    import torch
    from torch import nn
    from transformers import AutoModel, AutoTokenizer

    from_cuda = str(device).startswith("cuda")
    dtype = torch.float16 if from_cuda else torch.float32
    tok = AutoTokenizer.from_pretrained(model_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModel.from_pretrained(model_dir, torch_dtype=dtype)
    base.config.use_cache = False
    if use_lora:
        try:
            from peft import LoraConfig, get_peft_model
        except ImportError:
            use_lora = False
    for p in base.parameters():
        p.requires_grad = False
    if use_lora:
        cfg = LoraConfig(r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05,
                         target_modules=LORA_TARGETS)
        base = get_peft_model(base, cfg)
        for p in base.parameters():  # adapters em fp32 (estável com GradScaler)
            if p.requires_grad:
                p.data = p.data.float()
    else:
        inner = base
        for mod in list(inner.layers[-2:]) + [inner.norm]:
            mod.float()
            for p in mod.parameters():
                p.requires_grad = True
    if grad_ckpt and from_cuda:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    hidden = base.config.hidden_size
    head = nn.Linear(hidden, n_classes)
    return tok, base.to(device), head.to(device), use_lora


class TextClassifier:
    def __init__(self, tok, base, head, classes, max_len, device):
        self.tok, self.base, self.head = tok, base, head
        self.classes, self.max_len, self.device = list(classes), max_len, device

    def logits(self, texts: Sequence[str]):
        import torch

        enc = self.tok(list(texts), padding=True, truncation=True, max_length=self.max_len,
                       return_tensors="pt").to(self.device)
        cuda = str(self.device).startswith("cuda")
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            h = self.base(**enc).last_hidden_state
        pooled = masked_mean(h.float(), enc["attention_mask"])
        return self.head(pooled)


def train_classifier(
    texts: Sequence[str],
    labels: Sequence[str],
    model_dir: str | Path,
    *,
    epochs: int = 3,
    batch: int = 8,
    accum: int = 2,
    lr: float = 2e-4,
    head_lr: float = 1e-3,
    max_len: int = 384,
    lora_r: int = 16,
    use_lora: bool = True,
    grad_ckpt: bool = True,
    seed: int = 0,
    device: str | None = None,
    log: Callable[[str], object] | None = print,
) -> TextClassifier:
    import torch
    from torch import nn

    torch.manual_seed(seed)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    classes = sorted(set(labels))
    y = torch.tensor([classes.index(v) for v in labels])
    tok, base, head, used_lora = _build(model_dir, len(classes), device, use_lora, lora_r,
                                        grad_ckpt)
    clf = TextClassifier(tok, base, head, classes, max_len, device)
    params = [p for p in base.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": params, "lr": lr},
                             {"params": head.parameters(), "lr": head_lr}], weight_decay=0.01)
    steps = epochs * math.ceil(len(texts) / (batch * accum))
    warm = max(1, steps // 10)

    def sched(s):
        return min(1.0, (s + 1) / warm) * max(0.0, (steps - s) / max(1, steps - warm))

    lrs = torch.optim.lr_scheduler.LambdaLR(opt, sched)
    cuda = str(device).startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    rng = np.random.default_rng(seed)
    base.train()
    head.train()
    t0, skipped = time.time(), 0
    for ep in range(epochs):
        order = rng.permutation(len(texts))
        total, n = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for b, s in enumerate(range(0, len(texts), batch)):
            idx = order[s : s + batch]
            out = clf.logits([texts[i] for i in idx])
            loss = loss_fn(out.float(), y[idx].to(device))
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / accum).backward()
            total, n = total + float(loss.detach()), n + 1
            if (b + 1) % accum == 0 or s + batch >= len(texts):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                lrs.step()
        if log:
            log(f"[{device}] época {ep + 1}/{epochs} loss={total / max(n, 1):.4f} "
                f"lora={used_lora} pulados={skipped} {time.time() - t0:.0f}s")
    base.eval()
    head.eval()
    return clf


def predict_proba(clf: TextClassifier, texts: Sequence[str], batch: int = 16) -> np.ndarray:
    import torch

    order = np.argsort([len(t) for t in texts])
    out = np.empty((len(texts), len(clf.classes)), dtype=np.float64)
    with torch.no_grad():
        for s in range(0, len(texts), batch):
            idx = order[s : s + batch]
            p = torch.softmax(clf.logits([texts[i] for i in idx]).float(), 1).cpu().numpy()
            out[idx] = p
    bad = ~np.isfinite(out).all(1)
    out[bad] = 1.0 / out.shape[1]  # fp16 estourou: previsão neutra
    return out

In [ ]:
# v3: 4 visões de TF-IDF + embeddings de um LLM congelado (Qwen3-Embedding, anexado em Input),
# misturados em log-probabilidade. O peso da mistura e o C do classificador de embeddings são
# escolhidos por CV agrupada por prompt aqui mesmo, sobre o treino (determinístico).
import os
import time

import numpy as np
from sklearn.metrics import balanced_accuracy_score

t0 = time.time()
train, test = load_train(), load_test()
Xtr, ytr = train["text"].to_numpy(), train["model"].to_numpy()
Xte = test["text"].fillna("").to_numpy()
folds = grouped_folds(train, 5)


def lg(P):
    return np.log(np.clip(P, 1e-9, 1))


def ba(P):
    return round(float(balanced_accuracy_score(ytr, classes[P.argmax(1)])), 4)


P1, classes = oof_proba(four_view_lr, Xtr, ytr, folds)
T1 = four_view_lr().fit(Xtr, ytr).predict_proba(Xte)
print("4 visões CV", ba(P1), f"{time.time() - t0:.0f}s", flush=True)

texts = [str(t) for t in Xtr] + [str(t) for t in Xte]
if os.environ.get("NTL_FAKE_EMBED"):
    E = fake_embed(texts)
else:
    mdir = find_model_dir("/kaggle/input", os.environ.get("NTL_EMB", "qwen-3-embedding"))
    assert mdir is not None, "anexe um modelo Qwen3-Embedding em Input (Add Input > Models)"
    print("modelo", mdir, flush=True)
    E = embed_texts(texts, mdir)
Etr, Ete = E[: len(Xtr)], E[len(Xtr) :]
print("embeddings", E.shape, f"{time.time() - t0:.0f}s", flush=True)

best = None
for C in (0.01, 0.05, 0.2):
    P2, c2 = oof_proba(lambda C=C: emb_lr(C), Etr, ytr, folds)
    assert list(c2) == list(classes)
    print(f"embeddings C={C} CV", ba(P2), flush=True)
    if best is None or ba(P2) > ba(best[1]):
        best = (C, P2)
C, P2 = best
T2 = emb_lr(C).fit(Etr, ytr).predict_proba(Ete)
w, s = choose_weight(lg(P2), lg(P1), ytr, classes)
print(f"mistura: peso embeddings={w} CV={s:.4f}", f"{time.time() - t0:.0f}s", flush=True)
pred = classes[blend_logp([lg(T2), lg(T1)], [w, 1 - w]).argmax(1)]
path = write_submission(make_submission(test["id"].tolist(), pred.tolist()))
print(path, len(test), "linhas")